In [0]:
from pyspark.sql.functions import (
    col, count, count_distinct, approx_count_distinct,
    sum as spark_sum, round as spark_round,
)

silver = spark.table("workspace.coffee.silver_order_lines")
products = spark.table("workspace.coffee.products")

In [0]:
# task 10
by_channel = (silver
    .groupBy("channel")
    .agg(count("*").alias("lines"),
         count_distinct("order_id").alias("orders"),
         spark_round(spark_sum("line_amount_eur"), 2).alias("revenue_eur"))
    .orderBy("channel"))

by_channel.write.mode("overwrite").saveAsTable("workspace.coffee.gold_revenue_by_channel")
display(by_channel)

In [0]:
# task 11
by_product = (silver
    .join(products, on="product", how="left")
    .groupBy("product", "category")
    .agg(spark_sum("qty").alias("qty"),
         spark_round(spark_sum("line_amount_eur"), 2).alias("revenue_eur"),
         approx_count_distinct("customer_id").alias("customers"))
    .orderBy(col("revenue_eur").desc()))

by_product.write.mode("overwrite").saveAsTable("workspace.coffee.gold_product_revenue")
display(by_product)

In [0]:
%sql
-- task 11, step 4
SELECT category, ROUND(SUM(revenue_eur), 2) AS revenue_eur
FROM workspace.coffee.gold_product_revenue
GROUP BY category
ORDER BY revenue_eur DESC;